In [1]:

from typing import List, Tuple
import re
import os
import torch
import numpy as np


N = 10000
MAX_EPOCHS = 50
WEIGHT_DECAY = 0.05
LR = 1e-4
WARMUP = int(0.1 * MAX_EPOCHS)
N_RUNS = 100
NAMES = ["conch", "uni", "uni2h", "clipvitbasepatch32"]
EMBED = [512, 1024, 1536, 512]

In [2]:
path_ = "/Users/miguelmartins/Projects/thunder-identifiable/logs_da"

In [3]:
experiments = os.listdir(path_)
print(experiments)
logs = {}
paths = [os.path.join(path_, name) for name in experiments]
for i, name in enumerate(experiments):
    split_ = name.split("_")
    logs[f"{split_[0]}_{split_[1]}"] = paths[i]

['clipvitbasepatch32_tcga_tils', 'uni2h_esca', 'uni2h_tcga_tils', 'clipvitbasepatch32_esca', 'uni2h_crc', 'conch_crc', 'conch_esca', 'clipvitbasepatch32_crc', 'uni_tcga_tils', 'uni_esca', 'uni_crc', 'conch_tcga_tils']


In [4]:
for x, y in logs.items():
    print(x, y)

clipvitbasepatch32_tcga /Users/miguelmartins/Projects/thunder-identifiable/logs_da/clipvitbasepatch32_tcga_tils
uni2h_esca /Users/miguelmartins/Projects/thunder-identifiable/logs_da/uni2h_esca
uni2h_tcga /Users/miguelmartins/Projects/thunder-identifiable/logs_da/uni2h_tcga_tils
clipvitbasepatch32_esca /Users/miguelmartins/Projects/thunder-identifiable/logs_da/clipvitbasepatch32_esca
uni2h_crc /Users/miguelmartins/Projects/thunder-identifiable/logs_da/uni2h_crc
conch_crc /Users/miguelmartins/Projects/thunder-identifiable/logs_da/conch_crc
conch_esca /Users/miguelmartins/Projects/thunder-identifiable/logs_da/conch_esca
clipvitbasepatch32_crc /Users/miguelmartins/Projects/thunder-identifiable/logs_da/clipvitbasepatch32_crc
uni_tcga /Users/miguelmartins/Projects/thunder-identifiable/logs_da/uni_tcga_tils
uni_esca /Users/miguelmartins/Projects/thunder-identifiable/logs_da/uni_esca
uni_crc /Users/miguelmartins/Projects/thunder-identifiable/logs_da/uni_crc
conch_tcga /Users/miguelmartins/Proj

In [5]:
import tensorboard as tb

In [6]:
import pandas as pd

In [7]:
from glob import glob
from tensorboard.backend.event_processing import event_accumulator

In [8]:
def get_tensorboard_logs(
    log_dir: str, valid_metrics
):
    # vibe coded with o4-mini-high
    # find all event files under log_dir
    pattern = os.path.join(log_dir, "**", "events.out.tfevents.*")
    event_files = glob(pattern, recursive=True)
    if not event_files:
        raise FileNotFoundError(f"No TensorBoard event files found in {log_dir}")
    # for simplicity, just load the first one (you can loop if you have multiple)
    ea = event_accumulator.EventAccumulator(
        event_files[0],
        size_guidance={  # reduce load time
            event_accumulator.SCALARS: 0,
        },
    )
    ea.Reload()
    scalars = ea.Tags().get("scalars", [])
    if not scalars:
        raise RuntimeError("No scalar tags found in the event file.")
    valid_scalars = [scalar for scalar in scalars if scalar in valid_metrics]
    return_dict = {}
    steps = 0
    for tag in valid_scalars:
        events = ea.Scalars(tag)
        steps = [e.step for e in events]
        values = [e.value for e in events]
        return_dict[tag] = values
    return return_dict, steps

In [48]:
valid_augs = ['h_flip', 'crop', 'gauss', 'jitter']
tb_logs = {}
for model in logs.keys():
    it = logs[model]
    tb_logs[model] = dict()
    for aug in valid_augs:
        try:
            date_ = [x for x in os.listdir(f"{it}/{aug}") if x != "checkpoints"][0]
            tb_ = [x for x in os.listdir(f"{it}/{aug}/{date_}") if x != "hparams.yaml"][0]        
            tb_logs[model][aug] = {"path": f"{it}/{aug}"}
            metrics, _ = get_tensorboard_logs(tb_logs[model][aug]["path"], ['train_loss', 'id_acc'])
            
            tb_logs[model][aug] = metrics
        except:
            continue


#get_tensorboard_logs(f"{it}/h_flip/2025-10-12_09-43-44", ['train_loss', 'id_acc'])

In [49]:
df_logs = pd.DataFrame.from_dict(tb_logs, orient="index")

In [50]:
df_logs

,h_flip,crop,gauss,jitter
clipvitbasepatch32_tcga,"{'train_loss': [6.958677768707275, 6.959775924...","{'train_loss': [6.942624092102051, 6.876873493...","{'train_loss': [6.90261173248291, 6.9278049468...","{'train_loss': [6.962040901184082, 6.910572052..."
uni2h_esca,"{'train_loss': [6.954167366027832, 6.906890392...","{'train_loss': [6.925414085388184, 6.937306404...","{'train_loss': [6.9659576416015625, 6.80791568...","{'train_loss': [6.862420082092285, 6.895195960..."
uni2h_tcga,"{'train_loss': [6.887886047363281, 6.905303478...","{'train_loss': [6.954281806945801, 6.897325515...","{'train_loss': [6.9232330322265625, 6.91384840...","{'train_loss': [6.9620208740234375, 6.79437160..."
clipvitbasepatch32_esca,"{'train_loss': [6.948883056640625, 6.871221542...","{'train_loss': [6.961685657501221, 6.902503967...","{'train_loss': [6.948285102844238, 6.940958499...","{'train_loss': [6.869380950927734, 6.956393241..."
uni2h_crc,"{'train_loss': [6.950680732727051, 6.980272293...","{'train_loss': [6.8984551429748535, 6.90217447...","{'train_loss': [6.857708930969238, 6.885945320...","{'train_loss': [7.001880645751953, 6.930299758..."
conch_crc,"{'train_loss': [6.969566822052002, 6.987078666...","{'train_loss': [7.110743522644043, 7.027651786...","{'train_loss': [7.171401023864746, 6.996041297...","{'train_loss': [7.079451084136963, 6.986544132..."
conch_esca,"{'train_loss': [7.258676052093506, 6.953694820...","{'train_loss': [7.102358818054199, 6.897910118...","{'train_loss': [6.952624320983887, 6.875587463...","{'train_loss': [7.011757850646973, 6.937551021..."
clipvitbasepatch32_crc,"{'train_loss': [6.899087429046631, 6.882369041...","{'train_loss': [6.942008018493652, 6.932114601...","{'train_loss': [6.966984272003174, 6.882716655...","{'train_loss': [6.938930511474609, 6.915528774..."
uni_tcga,"{'train_loss': [7.049260139465332, 6.638995170...","{'train_loss': [7.04141902923584, 7.0546355247...","{'train_loss': [7.084367752075195, 6.996420860...","{'train_loss': [6.953338623046875, 6.762637615..."
uni_esca,"{'train_loss': [7.096766471862793, 6.818894386...","{'train_loss': [7.043174743652344, 6.908300876...","{'train_loss': [7.071558952331543, 6.878067493...","{'train_loss': [7.052755355834961, 6.639087677..."


In [51]:
import numpy as np

In [58]:
df_acc = df_logs.copy(deep=True)
for row in df_acc.index:
    augs = list(df_acc.loc[row].index)
    for aug in augs:
        try:
            acc = df_acc.loc[row][aug]['id_acc']
            df_acc.loc[row, aug] = np.mean(acc[-3:])
        except:
            continue

In [76]:
df_loss = df_logs.copy(deep=True)
for row in df_loss.index:
    augs = list(df_loss.loc[row].index)
    for aug in augs:
        try:
            acc = df_loss.loc[row][aug]['train_loss']
            df_loss.loc[row, aug] = np.mean(acc[-3:])
        except:
            continue

In [78]:
print(df_acc.apply(lambda s: s.mean(), axis=1)), "\n", print(df_loss.apply(lambda s: s.mean(), axis=1))

clipvitbasepatch32_tcga    0.665365
uni2h_esca                 0.973958
uni2h_tcga                 0.968750
clipvitbasepatch32_esca    0.696615
uni2h_crc                  0.979167
conch_crc                  0.757812
conch_esca                 0.787240
clipvitbasepatch32_crc     0.749740
uni_tcga                   0.958854
uni_esca                   0.960417
uni_crc                    0.958333
conch_tcga                 0.753906
dtype: float64
clipvitbasepatch32_tcga    6.491546
uni2h_esca                 0.790273
uni2h_tcga                 0.876299
clipvitbasepatch32_esca    6.141423
uni2h_crc                  0.901772
conch_crc                  4.491597
conch_esca                 4.485918
clipvitbasepatch32_crc     6.044876
uni_tcga                   0.821631
uni_esca                   0.462515
uni_crc                    0.654755
conch_tcga                 4.926443
dtype: float64


(None, '\n', None)

In [139]:
def get_wide(df_o):
    df = df_o.copy(deep=True)
    pairs = df.index.to_series().str.rsplit("_", n=1, expand=True)
    pairs.columns = ["model", "dataset"]
    
    # Make it a MultiIndex
    df.index = pd.MultiIndex.from_frame(pairs, names=["model", "dataset"])
    
    # Rows = model, Columns = dataset
    wide = df.unstack("dataset")
    wide.columns = wide.columns.set_names(['aug', 'dataset'])

    return wide
df_acc_wide = get_wide(df_acc)
df_loss_wide = get_wide(df_loss)

In [140]:
df_acc_wide

aug                h_flip                          crop                      \
dataset               crc      esca      tcga       crc      esca      tcga   
model                                                                         
clipvitbasepatch32    1.0  0.984375  0.986458  0.236458  0.234375  0.115625   
conch                 1.0  0.994792  0.994792     0.325  0.438542  0.307292   
uni                   1.0       1.0       1.0  0.833333  0.841667  0.835417   
uni2h                 1.0       1.0       1.0  0.916667  0.895833     0.875   

aug                    gauss                        jitter                      
dataset                  crc      esca      tcga       crc      esca      tcga  
model                                                                           
clipvitbasepatch32    0.8875  0.794792  0.780208     0.875  0.772917  0.779167  
conch               0.898958  0.873958  0.866667  0.807292  0.841667  0.846875  
uni                      1.0       1.0       1.0       1.0       1.0       1.0  
uni2h                    1.0       1.0       1.0       1.0       1.0       1.0

In [141]:
df_loss_wide

aug                   h_flip                          crop            \
dataset                  crc      esca      tcga       crc      esca   
model                                                                  
clipvitbasepatch32  5.785782  5.936338  6.371247  6.491932  6.407114   
conch               3.647273  3.836294   4.36108  5.506505  5.237172   
uni                 0.128402  0.101808  0.129863  2.005059  1.444438   
uni2h               0.239546  0.232763  0.265605  2.574747  2.404153   

aug                              gauss                        jitter  \
dataset                 tcga       crc      esca      tcga       crc   
model                                                                  
clipvitbasepatch32  6.659827  5.931229  6.064737   6.46573  5.970561   
conch               5.762047  4.304053  4.439493  4.766926  4.508557   
uni                 2.777132  0.239503  0.121731  0.197363  0.246055   
uni2h                2.72893  0.411323  0.198041  0.269984  0.381473   

aug                                     
dataset                 esca      tcga  
model                                   
clipvitbasepatch32  6.157502  6.469381  
conch               4.430714  4.815721  
uni                 0.182083  0.182168  
uni2h               0.326136  0.240678

In [169]:
lvl_aug = 0

styler = df_acc_wide.style.format('{:.3f}')
for aug in df_acc_wide.columns.get_level_values(lvl_aug).unique():
    cols = df_acc_wide.columns[df_acc_wide.columns.get_level_values(lvl_aug) == aug]
    styler = styler.background_gradient(
        subset=(slice(None), cols),
        cmap='RdYlGn',      # red->yellow->green
        vmin=0.0, vmax=1.0, # consistent across blocks
        low=0.1, high=0.1   # boost contrast near ends (optional)
    )
styler

In [150]:
lvl_aug = 0  # column level for 'aug'

styler = df_loss_wide.style.format('{:.3f}')
for aug in df_loss_wide.columns.get_level_values(lvl_aug).unique():
    cols = df_loss_wide.columns[
        df_loss_wide.columns.get_level_values(lvl_aug) == aug
    ]
    # Green for low, red for high (within each augmentation block)
    styler = styler.background_gradient(
        subset=(slice(None), cols),
        cmap='RdYlGn_r',      # red->yellow->green
        
    )
styler

In [167]:
import pandas as pd

# Ensure column level names
df_aug = df_acc_wide.copy()
df_aug.columns = df_aug.columns.set_names(['aug', 'dataset'])

# ---- 1) Average per row (rightmost) ----
row_avg = df_aug.mean(axis=1)
avg_col = pd.DataFrame({('avg', 'aug_avg'): row_avg}, index=df_aug.index)
df_out = pd.concat([df_aug, avg_col], axis=1)


# ---- Optional styling (green good, red bad). Flip to 'RdYlGn_r' for loss. ----
styler = (df_out.style
          .format('{:.3f}')
          .background_gradient(cmap='RdYlGn', axis=None)
          .set_properties(subset=(slice(None), [('avg','aug_avg')]), **{'font-weight':'bold'})
          #.set_properties(subset=(['avg_per_aug'], slice(None)), **{'font-weight':'bold'})
         )

styler


In [171]:
df_aug = df_acc_wide.copy()
df_aug.columns = df_aug.columns.set_names(['aug', 'dataset'])

# ---- 1) Column averages (per (aug,dataset)) as the FIRST row ----
col_avg = df_aug.mean(axis=0)  # one value per column
top = pd.DataFrame([col_avg.values], columns=df_aug.columns, index=['avg_per_col'])
df_out = pd.concat([top, df_aug], axis=0)

# ---- 2) Row averages (rightmost column), computed ONLY from original rows ----
row_avg = df_aug.mean(axis=1)  # don't include the top summary row
row_avg_all = pd.Series(index=df_out.index, dtype=float)
row_avg_all.loc[df_aug.index] = row_avg
row_avg_all.loc['avg_per_col'] = col_avg.mean()  # or use pd.NA if you prefer blank

df_out[('avg', 'row_avg')] = row_avg_all

# ---- Styling (green=better; use 'RdYlGn_r' for loss) ----
styler = (df_out.style
          .format('{:.3f}')
          .background_gradient(cmap='RdYlGn', axis=None)
          .set_properties(subset=(['avg_per_col'], slice(None)), **{'font-weight': 'bold'})
          .set_properties(subset=(slice(None), [('avg','row_avg')]), **{'font-weight':'bold'}))

styler

/var/folders/cr/qd8xdw991dv9ypjzzwsktpqc0000gn/T/ipykernel_36722/2159911247.py:12: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.7039062500827842 0.7663194429543284 0.959201388888889
 0.9739583333333334]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  row_avg_all.loc[df_aug.index] = row_avg


In [176]:
import pandas as pd

# Start from your wide df
df_aug = df_acc_wide.copy()
df_aug.columns = df_aug.columns.set_names(['aug', 'dataset'])

# -------- A) per-row average per augmentation (h_flip, crop, gauss, jitter, …) --------
lvl_aug = 0
aug_order = list(df_aug.columns.get_level_values(lvl_aug).unique())

row_by_aug = df_aug.groupby(level=lvl_aug, axis=1).mean()   # one mean per aug for each row
row_by_aug = row_by_aug[aug_order]                           # keep original aug order
# Put these under a new top-level 'avg'
row_by_aug.columns = pd.MultiIndex.from_product([['avg'], row_by_aug.columns])

# -------- B) overall per-row average (across all augs/datasets) --------
row_overall = pd.DataFrame({('avg', 'aug_avg'): df_aug.mean(axis=1)}, index=df_aug.index)

# -------- C) stitch together: original + avg block on the right --------
avg_block = pd.concat([row_by_aug, row_overall], axis=1)
df_out = pd.concat([df_aug, avg_block], axis=1)

# -------- D) style (green=better; use 'RdYlGn_r' for loss) --------
styler = (df_out.style
          .format('{:.3f}')
          .background_gradient(cmap='RdYlGn', axis=None)
          .set_properties(subset=(slice(None), [('avg','aug_avg')]), **{'font-weight':'bold'}))

styler


/var/folders/cr/qd8xdw991dv9ypjzzwsktpqc0000gn/T/ipykernel_36722/2377944538.py:11: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  row_by_aug = df_aug.groupby(level=lvl_aug, axis=1).mean()   # one mean per aug for each row


In [170]:
lvl_aug = 0

styler = df_out.style.format('{:.3f}')
for aug in df_out.columns.get_level_values(lvl_aug).unique():
    cols = df_out.columns[df_out.columns.get_level_values(lvl_aug) == aug]
    styler = styler.background_gradient(
        subset=(slice(None), cols),
        cmap='RdYlGn',      # red->yellow->green
        vmin=0.0, vmax=1.0, # consistent across blocks
        low=0.1, high=0.1   # boost contrast near ends (optional)
    )
styler

In [177]:
import pandas as pd

# Start from your wide df
df_aug = df_loss.copy()
df_aug.columns = df_aug.columns.set_names(['aug', 'dataset'])

# -------- A) per-row average per augmentation (h_flip, crop, gauss, jitter, …) --------
lvl_aug = 0
aug_order = list(df_aug.columns.get_level_values(lvl_aug).unique())

row_by_aug = df_aug.groupby(level=lvl_aug, axis=1).mean()   # one mean per aug for each row
row_by_aug = row_by_aug[aug_order]                           # keep original aug order
# Put these under a new top-level 'avg'
row_by_aug.columns = pd.MultiIndex.from_product([['avg'], row_by_aug.columns])

# -------- B) overall per-row average (across all augs/datasets) --------
row_overall = pd.DataFrame({('avg', 'aug_avg'): df_aug.mean(axis=1)}, index=df_aug.index)

# -------- C) stitch together: original + avg block on the right --------
avg_block = pd.concat([row_by_aug, row_overall], axis=1)
df_out = pd.concat([df_aug, avg_block], axis=1)

# -------- D) style (green=better; use 'RdYlGn_r' for loss) --------
styler = (df_out.style
          .format('{:.3f}')
          .background_gradient(cmap='RdYlGn', axis=None)
          .set_properties(subset=(slice(None), [('avg','aug_avg')]), **{'font-weight':'bold'}))

styler


/var/folders/cr/qd8xdw991dv9ypjzzwsktpqc0000gn/T/ipykernel_36722/2204787448.py:11: FutureWarning: DataFrame.groupby with axis=1 is deprecated. Do `frame.T.groupby(...)` without axis instead.
  row_by_aug = df_aug.groupby(level=lvl_aug, axis=1).mean()   # one mean per aug for each row


In [180]:
lvl_aug = 0

styler = df_out.style.format('{:.3f}')
for aug in df_out.columns.get_level_values(lvl_aug).unique():
    cols = df_out.columns[df_out.columns.get_level_values(lvl_aug) == aug]
    styler = styler.background_gradient(
        subset=(slice(None), cols),
        cmap='RdYlGn_r',      # red->yellow->green
    )
styler